In [3]:
import os
import nd2
import h5py
import numpy as np
from tqdm import tqdm
import ipywidgets as widgets
from IPython.display import display

def convert_nd2_to_hdf5(nd2_path, h5_path, dataset_name="frames", normalize_to_8bit=True, lut_min=None, lut_max=None, start_frame=0, end_frame=None, crop=None):
    """
    crop: optional (y_min, y_max, x_min, x_max) region applied to every frame.
          Same pixel box is cut out of each frame before it's written to the HDF5 file.
          Leave as None to keep the full frame.
    """
    if not os.path.exists(nd2_path):
        raise FileNotFoundError(f"Cannot find the ND2 file: {nd2_path}")

    print(f"\nOpening {nd2_path}...")
    
    with nd2.ND2File(nd2_path) as f:
        print(f"Original ND2 shape: {f.shape}")
        print(f"ND2 dtype: {f.dtype}")
        
        # MAC OPTIMIZATION: Always use lazy loading via dask to protect system memory
        video_data = f.to_dask()
        
        # Safely extract core spatial/temporal shapes
        total_frames = f.sizes.get('T', f.shape[0])
        height = f.sizes.get('Y', f.shape[-2])
        width = f.sizes.get('X', f.shape[-1])
        
        start_frame = max(0, start_frame)
        end_frame = total_frames if end_frame is None else min(end_frame, total_frames)
        target_num_frames = end_frame - start_frame
        
        if target_num_frames <= 0:
            raise ValueError(f"Invalid frame range: Start ({start_frame}) must be less than End ({end_frame}).")

        # --- Resolve crop box (same box applied to every frame) ---
        if crop is not None:
            y_min, y_max, x_min, x_max = crop
            y_min = max(0, int(y_min))
            x_min = max(0, int(x_min))
            y_max = min(height, int(y_max))
            x_max = min(width, int(x_max))
            if y_max <= y_min or x_max <= x_min:
                raise ValueError(f"Invalid crop region: {crop}")
            crop_height = y_max - y_min
            crop_width = x_max - x_min
            print(f"Cropping every frame to x[{x_min}:{x_max}] y[{y_min}:{y_max}]  ({crop_width}x{crop_height}px)")
        else:
            y_min, y_max, x_min, x_max = 0, height, 0, width
            crop_height, crop_width = height, width

        print(f"Target HDF5 shape: ({target_num_frames}, {crop_height}, {crop_width})  [Frames {start_frame} to {end_frame-1}]")

        used_min, used_max = 0, 255
        
        # --- Helper to force any weird frame shape down to an explicit 2D slice, then crop ---
        def extract_2d_frame(raw_data):
            sq_data = np.squeeze(raw_data)
            while len(sq_data.shape) > 2:
                sq_data = sq_data[0] # Grab the first channel/z-stack if multi-dimensional
            sq_data = sq_data.compute() if hasattr(sq_data, 'compute') else sq_data
            return sq_data[y_min:y_max, x_min:x_max]

        # --- High-Efficiency Contrast LUT Estimation ---
        if normalize_to_8bit:
            if lut_min is None or lut_max is None:
                print(f"Calculating global min/max across target range without loading full video to RAM...")
                g_min = float('inf')
                g_max = float('-inf')
                
                # Mac Safety Check: Sample up to 10 spaced frames rather than looping through thousands
                sample_step = max(1, target_num_frames // 10)
                for idx in range(start_frame, end_frame, sample_step):
                    frame_sample = extract_2d_frame(video_data[idx])
                    g_min = min(g_min, frame_sample.min())
                    g_max = max(g_max, frame_sample.max())
                
                used_min = lut_min if lut_min is not None else g_min
                used_max = lut_max if lut_max is not None else g_max
            else:
                used_min = lut_min
                used_max = lut_max
            
            print(f"LUT Parameters Applied -> Min: {used_min} | Max: {used_max}")

        # --- Streamlined Safe HDF5 Generation Block ---
        with h5py.File(h5_path, 'w') as h5f:
            target_dtype = np.uint8 if normalize_to_8bit else f.dtype
            
            dataset = h5f.create_dataset(
                dataset_name, 
                shape=(target_num_frames, crop_height, crop_width), 
                dtype=target_dtype,
                chunks=(1, crop_height, crop_width), # Chunked per frame for rapid streaming reads
                compression="gzip", 
                compression_opts=4
            )
            
            print(f"Writing chunks sequentially to {h5_path}...")
            out_idx = 0
            
            # Use progress bar to stream chunks one-by-one safely down onto disk storage
            for i in tqdm(range(start_frame, end_frame), desc="Converting Frames"):
                frame = extract_2d_frame(video_data[i])
                
                if normalize_to_8bit and frame.dtype != np.uint8:
                    frame_float = frame.astype(np.float32)
                    frame_float = np.clip(frame_float, used_min, used_max)
                    
                    if used_max > used_min:
                        frame = ((frame_float - used_min) / (used_max - used_min)) * 255.0
                    else:
                        frame = frame_float - used_min
                    frame = frame.astype(np.uint8)
                
                dataset[out_idx, :, :] = frame
                out_idx += 1

    print("Conversion complete!")
    if normalize_to_8bit:
        print(f"--- FINAL LUT SAVED -> Min: {used_min} | Max: {used_max} ---\n")

In [1]:
import os
import re
import csv
import nd2
import numpy as np
import pandas as pd
import tkinter as tk
import matplotlib.pyplot as plt
import ipywidgets as widgets
from tkinter import filedialog
from IPython.display import display, clear_output

# --- Paste your exact file path here ---
nd2_path = r"/Users/dharani/Desktop/sparse recordings/20260821_1_5_several untracked cases.nd2"

def interactive_preview(nd2_path):
    """
    One combined preview: pick a Frame, adjust LUT Min/Max, and drag out a
    crop box (X min/max, Y min/max) - all on the same image, updating live.
    Returns a dict that keeps updating as you move the sliders, so the
    conversion cell can just read it directly.
    """
    if not os.path.exists(nd2_path):
        print(f"Error: File not found at {nd2_path}")
        return None

    print(f"Loading preview for: {nd2_path}")
    f = nd2.ND2File(nd2_path)
    if 'T' in f.sizes:
        total_frames = f.sizes['T']
    else:
        total_frames = f.shape[0] if len(f.shape) > 2 else 1

    video_data = f.to_dask()

    def extract_2d_frame(raw_data):
        sq_data = np.squeeze(raw_data)
        while len(sq_data.shape) > 2:
            sq_data = sq_data[0]
        return sq_data.compute() if hasattr(sq_data, 'compute') else sq_data

    first_frame = extract_2d_frame(video_data[0] if len(video_data.shape) > 2 else video_data)
    height, width = first_frame.shape
    f.close()

    max_slider_val = int(first_frame.max() * 1.5)
    if max_slider_val <= 0:
        max_slider_val = 65535

    output_window = widgets.Output()
    params = {
        "frame_idx": 0,
        "lut_min": int(first_frame.min()),
        "lut_max": int(first_frame.max()),
        "x_min": 0, "x_max": width,
        "y_min": 0, "y_max": height,
    }

    frame_s = widgets.IntSlider(min=0, max=total_frames - 1, step=1, value=0, description='Frame:', continuous_update=False)
    lut_min_s = widgets.IntSlider(min=0, max=max_slider_val, step=1, value=params["lut_min"], description='LUT Min:', continuous_update=False)
    lut_max_s = widgets.IntSlider(min=1, max=max_slider_val, step=1, value=params["lut_max"], description='LUT Max:', continuous_update=False)
    x_min_s = widgets.IntSlider(min=0, max=width, step=1, value=0, description='X min:', continuous_update=False)
    x_max_s = widgets.IntSlider(min=0, max=width, step=1, value=width, description='X max:', continuous_update=False)
    y_min_s = widgets.IntSlider(min=0, max=height, step=1, value=0, description='Y min:', continuous_update=False)
    y_max_s = widgets.IntSlider(min=0, max=height, step=1, value=height, description='Y max:', continuous_update=False)

    all_sliders = [frame_s, lut_min_s, lut_max_s, x_min_s, x_max_s, y_min_s, y_max_s]

    # Guard flag so overlapping/duplicate change events can't trigger more than
    # one render pass at a time.
    _busy = {"flag": False}

    def render(*_args):
        if _busy["flag"]:
            return
        _busy["flag"] = True
        try:
            frame_idx = frame_s.value
            lut_min, lut_max = sorted((lut_min_s.value, lut_max_s.value))
            x_min, x_max = sorted((x_min_s.value, x_max_s.value))
            y_min, y_max = sorted((y_min_s.value, y_max_s.value))
            params.update(frame_idx=frame_idx, lut_min=lut_min, lut_max=lut_max,
                           x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max)

            raw_frame = video_data[frame_idx] if len(video_data.shape) > 2 else video_data
            frame = extract_2d_frame(raw_frame)

            frame_float = frame.astype(np.float32)
            frame_float = np.clip(frame_float, lut_min, lut_max)
            if lut_max > lut_min:
                frame_norm = ((frame_float - lut_min) / (lut_max - lut_min)) * 255.0
            else:
                frame_norm = frame_float - lut_min
            frame_uint8 = np.uint8(frame_norm)

            output_window.clear_output(wait=True)
            with output_window:
                fig, ax = plt.subplots(figsize=(8, 6))
                ax.imshow(frame_uint8, cmap='gray', vmin=0, vmax=255)
                rect = plt.Rectangle(
                    (x_min, y_min), max(x_max - x_min, 1), max(y_max - y_min, 1),
                    edgecolor='red', facecolor='none', linewidth=2
                )
                ax.add_patch(rect)
                ax.set_title(
                    f"Frame {frame_idx} | LUT [{lut_min}, {lut_max}] | "
                    f"Crop: x[{x_min}:{x_max}] y[{y_min}:{y_max}] ({x_max - x_min}x{y_max - y_min}px)"
                )
                ax.axis('off')
                plt.show()
                plt.close(fig)
                print(f"lut_min={lut_min}, lut_max={lut_max}")
                print(f"crop = ({y_min}, {y_max}, {x_min}, {x_max})  # (y_min, y_max, x_min, x_max)")
        finally:
            _busy["flag"] = False

    for s in all_sliders:
        s.observe(render, names='value')

    ui = widgets.VBox(all_sliders)
    display(ui, output_window)
    render()  # single initial draw

    return params  # same dict object, mutated live as sliders move

# Drag Frame to a good frame, LUT Min/Max to set contrast, and the X/Y sliders
# to box the region you want kept in every frame.
preview_params = interactive_preview(nd2_path)

Loading preview for: /Users/dharani/Desktop/sparse recordings/20260821_1_5_several untracked cases.nd2


Output()

In [4]:
# Set up your file targets manually 
input_nd2 = r"/Users/dharani/Desktop/sparse recordings/20260821_1_5_several untracked cases.nd2" 
output_h5 = r"/Users/dharani/Desktop/sparse recordings/20260821_1_5_several untracked cases.h5" 

# Everything below comes straight from the sliders in the preview cell above.
crop_region = (preview_params["y_min"], preview_params["y_max"], preview_params["x_min"], preview_params["x_max"])

# Run it!
convert_nd2_to_hdf5(
    nd2_path=input_nd2,
    h5_path=output_h5,
    normalize_to_8bit=True,
    lut_min=preview_params["lut_min"],
    lut_max=preview_params["lut_max"],
    start_frame=0, 
    end_frame=None,
    crop=crop_region
)


Opening /Users/dharani/Desktop/sparse recordings/20260821_1_5_several untracked cases.nd2...
Original ND2 shape: (281, 1562, 1110)
ND2 dtype: uint16
Cropping every frame to x[0:860] y[126:793]  (860x667px)
Target HDF5 shape: (281, 667, 860)  [Frames 0 to 280]
LUT Parameters Applied -> Min: 96 | Max: 135
Writing chunks sequentially to /Users/dharani/Desktop/sparse recordings/20260821_1_5_several untracked cases.h5...


Converting Frames: 100%|██████████| 281/281 [00:09<00:00, 30.40it/s]


Conversion complete!
--- FINAL LUT SAVED -> Min: 96 | Max: 135 ---

